In [2]:
import pandas as pd

df = pd.read_excel("personal_expenses.xlsx", header=None)

df = df.iloc[:, :3]
df.columns = ["Date", "Category", "Amount"]

df = df[df["Date"] != "Date"].copy()

df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Amount"] = pd.to_numeric(df["Amount"], errors="coerce")

df = df.dropna(subset=["Date", "Category", "Amount"])

print("Excel data imported successfully!")
print(df.head())
print(df.columns.tolist())

Excel data imported successfully!
        Date   Category  Amount
2 2026-01-05       Food   250.0
3 2026-01-07  Transport   100.0
4 2026-01-10   Shopping  1200.0
5 2026-01-12  Education   500.0
6 2026-01-15      Bills   800.0
['Date', 'Category', 'Amount']


In [4]:

import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

df["Date"] = pd.to_datetime(df["Date"])
df["Amount"] = pd.to_numeric(df["Amount"])
df["Month"] = df["Date"].dt.strftime("%B %Y")

categories = ["All"] + sorted(df["Category"].dropna().unique().tolist())
months = ["All"] + sorted(df["Month"].unique().tolist())

category_filter = widgets.Dropdown(
    options=categories,
    value="All",
    description="Category:"
)

month_filter = widgets.Dropdown(
    options=months,
    value="All",
    description="Month:"
)

budget_input = widgets.IntText(
    value=5000,
    description="Budget ₹:"
)

update_button = widgets.Button(
    description="Update Dashboard",
    button_style="primary",
    icon="refresh"
)

output = widgets.Output()

def show_dashboard(button=None):
    with output:
        clear_output(wait=True)

        data = df.copy()

        if category_filter.value != "All":
            data = data[data["Category"] == category_filter.value]

        if month_filter.value != "All":
            data = data[data["Month"] == month_filter.value]

        total = data["Amount"].sum()
        count = len(data)
        average = data["Amount"].mean() if count else 0
        budget = budget_input.value
        remaining = budget - total

        display(HTML("""
        <div style="background:linear-gradient(120deg,#172554,#2563eb);
        padding:25px;border-radius:15px;color:white;text-align:center">
        <h1 style="color:white;margin:0">Personal Expense Dashboard</h1>
        <p style="font-size:15px">Track • Analyze • Manage Your Money</p>
        </div>
        """))

        cards = [
            ("Total Spending", f"₹{total:,.2f}", "#2563eb"),
            ("Transactions", f"{count}", "#7c3aed"),
            ("Average Expense", f"₹{average:,.2f}", "#0891b2"),
            ("Budget Remaining", f"₹{remaining:,.2f}", "#059669" if remaining >= 0 else "#dc2626")
        ]

        card_html = '<div style="display:flex;flex-wrap:wrap;gap:12px;margin:15px 0">'
        for title, value, color in cards:
            card_html += f"""
            <div style="flex:1;min-width:150px;background:{color};
            padding:18px;border-radius:12px;color:white;text-align:center">
            <div style="font-size:14px">{title}</div>
            <div style="font-size:24px;font-weight:bold;margin-top:8px">{value}</div>
            </div>
            """
        card_html += "</div>"
        display(HTML(card_html))

        if data.empty:
            display(HTML("<h3>No expenses found for these filters.</h3>"))
            return

        category_data = data.groupby("Category")["Amount"].sum().sort_values(ascending=False)
        monthly_data = data.groupby("Month")["Amount"].sum()

        fig, axes = plt.subplots(2, 2, figsize=(13, 8))
        fig.suptitle("Expense Insights", fontsize=18, fontweight="bold")

        category_data.plot(kind="bar", ax=axes[0, 0], color="#3b82f6")
        axes[0, 0].set_title("Spending by Category")
        axes[0, 0].set_ylabel("Amount (₹)")
        axes[0, 0].tick_params(axis="x", rotation=35)

        monthly_data.plot(kind="line", marker="o", ax=axes[0, 1], color="#059669")
        axes[0, 1].set_title("Monthly Spending Trend")
        axes[0, 1].set_ylabel("Amount (₹)")
        axes[0, 1].tick_params(axis="x", rotation=35)

        category_data.plot(
            kind="pie", ax=axes[1, 0], autopct="%1.1f%%",
            startangle=90
        )
        axes[1, 0].set_title("Expense Distribution")
        axes[1, 0].set_ylabel("")

        axes[1, 1].barh(
            category_data.index, category_data.values, color="#f59e0b"
        )
        axes[1, 1].set_title("Category Comparison")
        axes[1, 1].set_xlabel("Amount (₹)")

        plt.tight_layout()
        plt.show()

        display(HTML("<h3 style='color:#2563eb'>Recent Transactions</h3>"))
        display(data.sort_values("Date", ascending=False).head(10))

        if remaining < 0:
            display(HTML("<h3 style='color:#dc2626'>Budget exceeded!</h3>"))
        else:
            display(HTML("<h3 style='color:#059669'>You are within your budget.</h3>"))

update_button.on_click(show_dashboard)

display(widgets.VBox([
    widgets.HTML("<h3>Dashboard Controls</h3>"),
    widgets.HBox([category_filter, month_filter]),
    widgets.HBox([budget_input, update_button]),
    output
]))

show_dashboard()
